# In-Domain Qualitative Figure (Colab)

This notebook rebuilds a publication-ready in-domain qualitative figure for the curated benchmark.

It compares:
- Ground truth
- Faster R-CNN lightweight reference (`baseline_mobilenet_hr_vsb7_3600_rarefirst_full`)
- YOLOv8s baseline (`y0_yolov8s_vsb7_3600_rarefirst_e200`)
- YOLO variant (`y1_yolov8s_p2_vsb7_3600_rarefirst_e200`)

The notebook assumes:
- the processed dataset root is available at `/content/processed_for_server`
- you want artifacts written to Google Drive
- you are using a repo snapshot that contains `scripts/build_in_domain_qualitative_figure.py` and the patched `scripts/evaluate_yolov8.py --save-predictions`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/ntkhanh98/wood-defect-q2.git"
REPO_DIR = Path('/content/wood-defect-q2')
DATA_ROOT = Path('/content/processed_for_server')
MAIN_ROOT = DATA_ROOT / 'main_dataset'
ARTIFACT_ROOT = Path('/content/drive/MyDrive/wood_q2_in_domain_artifacts')

ROWS = 5
IMAGE_SIZE = 1024
YOLO_EPOCHS = 200
TWO_STAGE_EPOCHS = 10  # matches the retained baseline_mobilenet_hr_vsb7_3600_rarefirst_full run
DEVICE = '0'

os.environ['WOOD_MAIN_PROCESSED_ROOT'] = str(MAIN_ROOT)
os.environ['PYTHONUNBUFFERED'] = '1'

print('REPO_DIR    =', REPO_DIR)
print('MAIN_ROOT   =', MAIN_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)


## 1. Clone repo and install dependencies

If you already uploaded the current repo snapshot to `/content/wood-defect-q2`, you can skip the clone cell and only run the install lines.

In [ ]:
%%bash
set -euo pipefail

cd /content
rm -rf wood-defect-q2
git clone "$REPO_URL" wood-defect-q2
cd wood-defect-q2

python3 -m pip install -q ultralytics==8.3.0 timm pycocotools pandas pillow pyyaml


In [ ]:
for p in [
    MAIN_ROOT / 'manifest.jsonl',
    MAIN_ROOT / 'metadata.json',
    REPO_DIR / 'scripts' / 'build_in_domain_qualitative_figure.py',
    REPO_DIR / 'scripts' / 'evaluate_yolov8.py',
]:
    print(p, 'OK' if p.exists() else 'MISSING')

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'yolo').mkdir(exist_ok=True)
(ARTIFACT_ROOT / 'tables').mkdir(exist_ok=True)
(ARTIFACT_ROOT / 'figures').mkdir(exist_ok=True)


## 2. Build the curated benchmark and YOLO export

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/build_screened_benchmark.py \
  --input-manifest /content/processed_for_server/main_dataset/manifest.jsonl \
  --output-root-dir /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first \
  --dataset-name large_scale_wood_surface_defects_vsb7_3600_rare_first \
  --target-source-images 3600 \
  --seed 42 \
  --classes live_knot dead_knot resin knot_with_crack crack marrow knot_missing \
  --selection-mode rare_first

python3 scripts/build_yolo_dataset.py \
  --input-manifest /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first/manifest.jsonl \
  --image-root-dir /content/processed_for_server/main_dataset \
  --output-root-dir /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first_yolo \
  --dataset-name large_scale_wood_surface_defects_vsb7_3600_rare_first_yolo \
  --classes live_knot dead_knot resin knot_with_crack crack marrow knot_missing


In [ ]:
for p in [
    MAIN_ROOT / 'benchmarks' / 'vsb7_3600_rare_first' / 'manifest.jsonl',
    MAIN_ROOT / 'benchmarks' / 'vsb7_3600_rare_first_yolo' / 'dataset.yaml',
]:
    print(p, 'OK' if p.exists() else 'MISSING')


## 3. Create generated configs and the Y1 model YAML

The current branch may not contain `configs/models/yolov8s-p2-7class.yaml`, so this cell writes it explicitly.

In [ ]:
import yaml
from textwrap import dedent

generated_dir = REPO_DIR / 'configs' / 'generated_colab'
models_dir = REPO_DIR / 'configs' / 'models'
generated_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

train_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(ARTIFACT_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full',
    'dataset': {
        'train': 'configs/dataset_main_vsb7_3600_rarefirst.yaml',
        'val': 'configs/dataset_main_vsb7_3600_rarefirst.yaml',
        'train_split': 'train',
        'val_split': 'val',
    },
    'dataset_split': {
        'seed': 42,
        'train_ratio': 0.8,
        'val_ratio': 0.1,
    },
    'train': {
        'epochs': int(TWO_STAGE_EPOCHS),
        'batch_size': 2,
        'num_workers': 2,
        'image_size': int(IMAGE_SIZE),
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'best_metric': 'mAP50_95',
        'small_defect_sampler': {
            'enabled': False,
            'small_weight': 3.0,
            'positive_weight': 1.5,
            'negative_weight': 0.5,
        },
    },
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
}

eval_cfg = {
    'seed': 42,
    'device': 'cuda',
    'output_dir': str(ARTIFACT_ROOT),
    'experiment_name': 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval',
    'checkpoint_path': str(ARTIFACT_ROOT / 'checkpoints' / 'baseline_mobilenet_hr_vsb7_3600_rarefirst_full' / 'best.pt'),
    'dataset': {
        'eval': 'configs/dataset_main_vsb7_3600_rarefirst.yaml',
        'split': 'test',
    },
    'dataset_split': {
        'seed': 42,
        'train_ratio': 0.8,
        'val_ratio': 0.1,
    },
    'model': {
        'name': 'baseline_detector',
        'num_classes': 7,
        'backbone': 'mobilenet_hr',
        'image_size': int(IMAGE_SIZE),
        'score_threshold': 0.05,
        'nms_threshold': 0.5,
        'max_detections': 100,
    },
    'evaluation': {
        'batch_size': 1,
        'num_workers': 2,
        'score_threshold': 0.05,
        'compute_small_defect_eval': True,
        'tile_merge': False,
        'save_predictions': True,
        'save_visualizations': False,
        'compute_per_class_ap': True,
        'compute_cross_dataset': False,
    },
}

train_cfg_path = generated_dir / 'train_baseline_vsb7_3600_rarefirst_full.yaml'
eval_cfg_path = generated_dir / 'eval_baseline_vsb7_3600_rarefirst_full.yaml'
with train_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(train_cfg, f, sort_keys=False)
with eval_cfg_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False)

y1_model_yaml = dedent('''\
# Ultralytics YOLOv8s-P2 detection model specialized for the 7-class wood-defect setup.
# Based on the official YOLOv8 P2 head layout, with explicit "s" scaling baked in.

nc: 7
depth_multiple: 0.33
width_multiple: 0.50

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 3, C2f, [128]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 12], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 9], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[18, 21, 24, 27], 1, Detect, [nc]]
''')
y1_model_path = models_dir / 'yolov8s-p2-7class.yaml'
y1_model_path.write_text(y1_model_yaml, encoding='utf-8')

print('Wrote', train_cfg_path)
print('Wrote', eval_cfg_path)
print('Wrote', y1_model_path)


## 4. Train and evaluate the Faster R-CNN reference

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train.py \
  --config configs/generated_colab/train_baseline_vsb7_3600_rarefirst_full.yaml \
  --experiment-name baseline_mobilenet_hr_vsb7_3600_rarefirst_full \
  --epochs 10 \
  --device cuda

python3 scripts/evaluate.py \
  --config configs/generated_colab/eval_baseline_vsb7_3600_rarefirst_full.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_in_domain_artifacts/checkpoints/baseline_mobilenet_hr_vsb7_3600_rarefirst_full/best.pt \
  --experiment-name baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval \
  --device cuda


## 5. Train and evaluate the YOLO baseline

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train_yolov8.py \
  --data /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first_yolo/dataset.yaml \
  --model yolov8s \
  --experiment-name y0_yolov8s_vsb7_3600_rarefirst_e200 \
  --epochs 200 \
  --imgsz 1024 \
  --batch 32 \
  --device 0 \
  --workers 4 \
  --seed 42 \
  --patience 50 \
  --project-dir /content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_main_vsb7_3600_rarefirst.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo/y0_yolov8s_vsb7_3600_rarefirst_e200/weights/best.pt \
  --experiment-name y0_yolov8s_vsb7_3600_rarefirst_e200_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_in_domain_artifacts \
  --small-defect-eval \
  --save-predictions


## 6. Train and evaluate the YOLO variant (`Y1`)

This notebook defaults to `Y1` because it is the most reliable branchless fallback for the qualitative figure. If you later want `Y2`, replace this section with the custom WNIoU branch workflow.

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train_yolov8.py \
  --data /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first_yolo/dataset.yaml \
  --model /content/wood-defect-q2/configs/models/yolov8s-p2-7class.yaml \
  --experiment-name y1_yolov8s_p2_vsb7_3600_rarefirst_e200 \
  --epochs 200 \
  --imgsz 1024 \
  --batch 16 \
  --device 0 \
  --workers 4 \
  --seed 42 \
  --patience 50 \
  --project-dir /content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_main_vsb7_3600_rarefirst.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo/y1_yolov8s_p2_vsb7_3600_rarefirst_e200/weights/best.pt \
  --experiment-name y1_yolov8s_p2_vsb7_3600_rarefirst_e200_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_in_domain_artifacts \
  --small-defect-eval \
  --save-predictions


## 7. Build the final in-domain qualitative figure

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/build_in_domain_qualitative_figure.py \
  --manifest /content/processed_for_server/main_dataset/benchmarks/vsb7_3600_rare_first/manifest.jsonl \
  --image-root-dir /content/processed_for_server/main_dataset \
  --split test \
  --rows 5 \
  --baseline-run-name baseline_mobilenet_hr_vsb7_3600_rarefirst_full \
  --baseline-header "Faster R-CNN" \
  --baseline-predictions /content/drive/MyDrive/wood_q2_in_domain_artifacts/tables/baseline_mobilenet_hr_vsb7_3600_rarefirst_full_eval_test_predictions.jsonl \
  --yolo-run-name y0_yolov8s_vsb7_3600_rarefirst_e200 \
  --yolo-header YOLOv8s \
  --yolo-predictions /content/drive/MyDrive/wood_q2_in_domain_artifacts/tables/y0_yolov8s_vsb7_3600_rarefirst_e200_eval_test_predictions.jsonl \
  --variant-run-name y1_yolov8s_p2_vsb7_3600_rarefirst_e200 \
  --variant-header "YOLO P2" \
  --variant-predictions /content/drive/MyDrive/wood_q2_in_domain_artifacts/tables/y1_yolov8s_p2_vsb7_3600_rarefirst_e200_eval_test_predictions.jsonl \
  --output-dir /content/drive/MyDrive/wood_q2_in_domain_artifacts/figures/in_domain_qualitative


In [ ]:
%%bash
set -euo pipefail

find /content/drive/MyDrive/wood_q2_in_domain_artifacts/figures/in_domain_qualitative -maxdepth 1 -type f | sort
